# Design issues with AI/ML Projects and Strategy for Deep learning Project
- Joseph Fodera

## Machine Learning Strategy


- After your first iteration through the training data and collecting results, you can decide to update your model in a number of ways:
  - Decide you want ot collect more data
  - You want to train your algorithm longer
  - Switch to a different optimization algorithm
  - Use regularization
  - Tune Hyperparameters
- There are some efficient strategies below that will help to move in the right direction.

## Tuning ML Process (Orthogonalization)


- Typical ML workflow:
  - fit training set on the cost function
  - fit dev set on the cost function
  - fit the test set
- Very rarely does the model perform well on all three steps
  - So we must fine tune the model at each step (each step has their own parameters that need to be tunes)

- Step 1:
  - If not a good result, we may want to focus on **getting more data** or **choosing a good (different) optimization algorithm**
- Step 2:
  - If not a good result, may want to use (or use more) **regularization**
- Step 3:
  - If not a good result, could get a **bigger dev set** or **choose a different model**

- Thinking of these remedies in the context of these three steps is sometime referred to as **orthogonalization**.
  - Basically, orthogonalization is: depending on the issue you are facing motivates your remedy.

## Evaluation Metric

- In some cases, selecting two evaluation metrics can be confusing given certain problems
  - Ex. Precision and Recall occasionally have a trade-off relationship
    - **Precision (Quality)** - of all instances the model predicted as positive, how many were actually positive?
    - **Recall (Quantity or Sensitivity)** - Of all the actual positive instances, how many did the model manage to find?
- So, for one model you might get a good result for one metric and vice versa.
- You can deal with this by employing an evaluation metric that encompasses both (or all) of your metrics.
  - In the case above you could use *F1 score* as it is a harmonic mean of precision and recall
    - *harmonic mean* forces both precision and racall to be high to get a good F-1 score, ensuring no cases where one is at 100% and the other is at 0% beause the model would fail



#### Constrained Optimization  


- An alternate way of combining metrics could be to formulate an optimization problem.
- This would be used to **evaluate the models ability to produce accurate predictions in a reasonable amount of time**.
- If both accuracy and runtime are important to your model, you can set a hard contraint on runtime and try to solve a maximizing accuracy problem with that constraint.
  - Would look something like: `Maximize: Accuracy (w) Subject to: Runtime(w) <= 100ms`
- This is very useful in large NN's that are achieving high accuracy at the expense of very high runtime
- some of the variables you can tune to balance accuracy and runtime include:
  - **Model architecture** - depth, width, or complexity of the model
  - **Hyperparameters** - # of layers, neurons, or features
  - **Feature Selection** - reducing the input dimensionality to save computation time
  - **Pruning/quantization** - reducing the size of the model at no cost to the performance/accuracy
- You can use the following techniques to help solve the constrained optimization problem:
  - *Grid Search:*
    - define iscrete values for hyperparameters or model settings
    - train and evaluate the model for all combinations
    - retain only configurations that satisfy the optimization objective (like a certain runtime constraint or something like that).
    - select model with the highest accuracy to solve the problem.
  - *Random Search:*
    - randomly sample configurations within your hyperparameter space
    - Evaluate A(M) and R(M) for each sampled configuration
      - *A(M)* - Accuracy of the Model
      - *R(M)* - Runtime of the Model      
    - Select the configuration with the best accuracy satisfying R(M) <= $R_{max}$
  - *Bayesian Optimization:*
    - Use a probabilistic model to explore the parameter space efficiently. Incorporate the runtime constraint into the optimization objective or as a hard constraint.
      - Another example of *meta learning* where 'probabilitic model' refers to a model, that goes on top of your model, with inputs as the hyperparameters of the base model, with its goal to optimize the output of the base model

## Hyperparameter Tuning (more in depth on techniques from above)


- The following are possible approaches for finding the optimal hyperparameters:
  1. Hand Tuning (aka trial and error)
    - Parameters chosen based on trial and error and the experience of the user.
  2. Grid Search
    - Grid is created based on parameter values.
    - All possible parameter combinations are tried.
    - Best combination is chosen
  3. Random Search
    - Instead of trying all possible combinations from grid search, we only select a random subset of the parameters to try.
    - The best subset is then chosen.
  4. Bayesian Optimization (Gausian Process)
    - Uses a set of previously evaluated parameters and resulting accuracy to make an assumption about unobserved parameters.
      - **Learns** based on previous trainings
    - The *Aquisition Function* uses this information leaned from the previous parameter attems to suggest the next set of parameters
  5. Tree-Structured Parzen Estimators (TPE)
    - Each iteration, TPI collects new observations and at the end of the iteration, the algorithm decides which set of parameters it should try next.

- An important aspect of hyperparameter tuning is that in most search based methods, the logs of the hyperparameters are sampled rather than the actual values.
  - For instance, if searching for η between 0.1 and 0.001, we first sample logη uniformly between -1 and -3
    - `log(0.001) = -3`
  - We then exponentiate to the power of 10
    - ex. $10^{log(0.001)}$
  - This allows us to search learning rate in 'orders of magnitude'
- Although, it is important to note that certain parameters are searched in the uniform space.
  - Simply means we sample values directly and evenly across a range.

## Feature Preprocessing

- Preprocessing in NN is not very different from other ML models
- **Additive preprocessing and mean centering**
  - It is useful to mean center the data to remove any kind of bias from the model.
    - *mean centering* - when you shift the data distribution of each feature respectivley so the mean of every feature becomes zero
  - Many algorthms like Principal Component Analysis (PCA) work with the assumption of mean-centered data.
  - Since, $$X_{centered} = X - \mu$$ we take a vector of column wise means and it is subtracted from each data point.
- Other preprocessing is done to get rid of negative values if this is desired.
  - We simply add the most negative entry to the rest of the values to ensure this
- **Feature Normalization**
  - Common practice is to divide each feature value by its standard deviation from the feature mean.
  - Combining this scaling with mean-centering results in **standardized** data.
    - This is because the data is then presumed to be drawn from a standard normal distribution with mean zero and unit variance.
  - Another kind is to compute the minimum and maximum value of any attribute.
    - Then, subtract the min from the data point and divide it by the difference between the max and min (do this for each data point).
  - Either way you do it, it ensures better performance as it is common for relative values of features to vary more than an order of magnitude.
    - Using these techniques helps us to lower the sensitivity of the learning algorithm for some features versus others (bias).


### Principal Component Analysis (PCA)


- Can be thought of as the application of Singular Value Decomposition (SVD) after mean-centering a data matrix.
- Take D as a nXm mean centered data matrix
- Take C as a nXn covariance matrix that gives you the covariance between dimensions
  - *covariance* - basically a raw measure of the correlation of features (how well them move with eachother)
- $C=(D^T D)/n$
- The eigenvectors of the covariance matrix (C) provide de-corrrelated directions in the data
  - In raw data, 'directions' are your features, but these are correlated.
  - Feature 1: Is just "Age.
  - "Eigenvector 1: Might be $(0.7 \times \text{Age}) + (0.2 \times \text{Income}) - (0.1 \times \text{Education})$.
  - But since these eigenvectors are ensured to be orthogonal (90deg) to eachother, the covariance between dimensions becomes zero
    - all redundancy between original features has been stripped away.

- Eigenvalues provide variance along each of the directions
  - a measure of magnitude of each eigenvector, representing the amount of variance associated with said combination of features that each decorrelated eigenvector is based off of.
- We then use the top-k eigenvectors (largest k eigenvalues) of the covariance matrix.
  - This results in most of the variance in the data (statistical significance) being retained while the noise is removed.
- One can choose some threshold eigenvalue when selecting these new dimensions.
  - The eigenvectors *are* the new dimentions, we are not just 'picking important features'.
"Let the final matrix be P which"



- **Whitening**
  - Technique of creating a new set of de-correlated features.
  - PCA is used to acieve this

#### SVD Base Definition


  - *SVD* - fundamental linear algebra theorem that says any matrix (no matter the shape) can be broken down into 3 specific and simpler pieces:
  - $$A = U \Sigma V^T$$
    - U (Left Sigular Vectors) - a matrix that represents the "geometry" of your rows.
      - In PCA, these are related to how your data points relate to eachother.
    - Σ (Singular Values) - Diagonal matrix of numbers sorted from largest to smallest
      - numbers tell you **how much information (variance)** is captured by each dimension
    - $V^T$ (Right Singular Vectors) - represents the 'geoetry' of your columns along with the *Principal Components* themselves.